# Project 3, Part 2, Queries on refugees staging table that we will use later to create the graph

## Included Modules and Packages

In [1]:
import csv

import math
import numpy as np
import pandas as pd

import psycopg2

## Supporting code

In [2]:
#
# function to run a select query and return rows in a pandas dataframe
# pandas puts all numeric values from postgres to float
# if it will fit in an integer, change it to integer
#

def my_select_query_pandas(query, rollback_before_flag, rollback_after_flag):
    "function to run a select query and return rows in a pandas dataframe"
    
    if rollback_before_flag:
        connection.rollback()
    
    df = pd.read_sql_query(query, connection)
    
    if rollback_after_flag:
        connection.rollback()
    
    # fix the float columns that really should be integers
    
    for column in df:
    
        if df[column].dtype == "float64":

            fraction_flag = False

            for value in df[column].values:
                
                if not np.isnan(value):
                    if value - math.floor(value) != 0:
                        fraction_flag = True

            if not fraction_flag:
                df[column] = df[column].astype('Int64')
    
    return(df)

In [3]:
connection = psycopg2.connect(
    user = "postgres",
    password = "ucb",
    host = "postgres",
    port = "5432",
    database = "postgres"
)

In [4]:
cursor = connection.cursor()

## Query the list of countries

In [5]:
rollback_before_flag = True
rollback_after_flag = True

query = """

(select coo as country
from stage_1_refugees)
union
(select coa as country
from stage_1_refugees)
order by country

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,country
0,ABW
1,AFG
2,AIA
3,ALB
4,ALG
...,...
207,WES
208,WSH
209,YEM
210,ZAM


## Query the list of refugee movements between countries for a particular year

In [6]:
rollback_before_flag = True
rollback_after_flag = True

query = """

select coo as from_country,
       coa as to_country,
       refugees
from stage_1_refugees
where (coo <> coa) and (refugees::numeric > 0) and (year::numeric = 2022)
order by coo, coa, refugees

"""

my_select_query_pandas(query, rollback_before_flag, rollback_after_flag)

,from_country,to_country,refugees
0,AFG,ALB,14
1,AFG,ARE,47
2,AFG,ARG,11
3,AFG,ARM,9
4,AFG,AUL,7787
...,...,...,...
4617,ZIM,SWE,14
4618,ZIM,SWI,9
4619,ZIM,THA,9
4620,ZIM,USA,680
